
## Was DeepEval uns bietet

- Fertige RAG-Metriken (keine Prompts zu schreiben):
  - **Faithfulness** — genau unser Faithful-Judge
  - **Answer Relevancy** — unser Relevant-Judge
  - **Contextual Relevancy** — war der *abgerufene Kontext* relevant?
    (DeepEvals Name für die **Context Precision**)
  - **Contextual Recall** — haben wir *genug* Kontext gefunden?
    (DeepEvals Name für den **Context Recall**: bewertet gegen die
    Referenzantwort — genau die praktische Definition aus der
    Dimensionen-Lektion)
- Für unseren **Correct-Judge** gibt es kein fertiges Gegenstück — dafür
  gibt es **GEval**: eine Metrik, der Sie das Kriterium selbst geben
- Wir führen **alle fünf** aus — die beiden Retrieval-Metriken, die unsere
  hand-built Judges ausgelassen haben, übernimmt hier das Framework
- Ein **Score von 0.0–1.0** *und* ein schriftlicher **Reason** pro Metrik
- Funktioniert mit jedem Modell — hier unser übliches OpenRouter-Modell


## DeepEval installieren

- Installation mit pip (Version festlegen — DeepEval entwickelt sich schnell weiter):

In [15]:
!pip install "deepeval>=4.2,<4.3"

Defaulting to user installation because normal site-packages is not writeable



## Judge-Modell konfigurieren

- DeepEval enthält ein `OpenRouterModel` — unsere übliche Einrichtung, eine Zeile Code reicht.

In [25]:
import os

# Muss vor dem Import von deepeval gesetzt werden; langsame OpenRouter-Antworten brauchen mehr als 88 s
os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "300")

from dotenv import load_dotenv
from deepeval.models import OpenRouterModel
from deepeval.test_case import LLMTestCase
from rag_utils import print_wrapped

load_dotenv()


True

In [26]:
class NoLogprobsOpenRouterModel(OpenRouterModel):
    # GEval fragt sonst Logprobs an, die das Modell über OpenRouter nicht liefert -> Hänger/Timeout
    def generate_raw_response(self, *args, **kwargs):
        raise AttributeError("log_probs unsupported.")


judge_model = NoLogprobsOpenRouterModel(
    model="ibm-granite/granite-4.2-8b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)



## Ein Test Case

- DeepEval kapselt jedes Beispiel in einem `LLMTestCase`
- Die Felder entsprechen den Daten, die wir bereits erfassen:
  - `input` — die Frage
  - `actual_output` — unsere RAG-Antwort
  - `retrieval_context` — die abgerufenen Dokumente (eine Liste)
  - `expected_output` — die Referenzantwort

In [18]:
good_case = LLMTestCase(
    input="Was ist Overfitting?",
    actual_output="Overfitting bedeutet, dass das Modell die Trainingsdaten auswendig lernt.",
    retrieval_context=[
        "Overfitting tritt auf, wenn ein Modell die Trainingsdaten auswendig lernt. "
        "Regularisierung hilft dagegen."
    ],
    expected_output="Das Modell lernt die Trainingsdaten auswendig und generalisiert nicht.",
)

In [19]:
# Ein halluzinierter Fall
# DeepEval sollte hier einen niedrigen Faithfulness-Score vergeben
bad_case = LLMTestCase(
    input="Was ist Overfitting?",
    actual_output="Overfitting wird durch eine zu niedrige Lernrate verursacht.",
    retrieval_context=[
        "Overfitting tritt auf, wenn ein Modell die Trainingsdaten auswendig lernt."
    ],
    expected_output="Das Modell lernt die Trainingsdaten auswendig und generalisiert nicht.",
)

In [27]:
from deepeval.metrics import FaithfulnessMetric
faithfulness = FaithfulnessMetric(model=judge_model, async_mode=False)
bad_score = faithfulness.measure(bad_case, _show_indicator=False)
assert bad_score is not None
assert bad_score == faithfulness.score
bad_reason = faithfulness.reason

good_score = faithfulness.measure(good_case, _show_indicator=False)
assert good_score is not None 
assert good_score == faithfulness.score
good_reason = faithfulness.reason


In [30]:
bad_score, good_score

(1.0, 1.0)

In [31]:
bad_reason, good_reason

('The score is 1.00 because there are no contradictions in the actual output relative to the retrieval context.',
 'The score is 1.00 because there are no contradictions in the actual output relative to the retrieval context.')


## Korrektheit mit GEval

- Faithfulness und Answer Relevancy sind fertige Metriken — unser dritter
  Judge (**correct?**) hat kein vorgefertigtes Gegenstück
- DeepEvals Antwort ist **GEval**: eine Metrik, der Sie das **Kriterium
  selbst geben** — wie bei unserem hand-built Judge, nur ohne Boilerplate
- `evaluation_params` legt fest, welche Felder der Judge sieht — dieselben
  wie bei unserem Correct-Judge: Frage, Antwort und Referenz

In [28]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

correctness = GEval(
    name="Correctness",
    criteria="The actual output is correct if it is consistent with the "
    "expected output. Minor wording differences are acceptable. If the "
    "expected output says the information is not available, the actual "
    "output should indicate the same.",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=judge_model,
    async_mode=False,
)

In [29]:
correctness.measure(good_case, _show_indicator=False)


0.7

In [32]:
correctness.score

0.7

In [33]:
print_wrapped(correctness.reason)

Actual Output correctly describes the model memorizing training data ('die
Trainingsdaten auswendig lernt'), aligning with 'lernt die Trainingsdaten
auswendig' in Expected Output. However, it omits 'und generalisiert nicht' from
Expected Output, which explicitly states the model does not generalize. Omitting
this key phrase exceeds minor wording differences allowed in Step 1, reducing
consistency with Expected Output.



## Mehrere Metriken, mehrere Fälle

- Iterieren Sie über unsere Fälle und Metriken und sammeln Sie die Scores in einer Tabelle
- `ContextualRelevancy` und `ContextualRecall` bewerten das **Retrieval**;
  Contextual Recall braucht dafür die Referenz (`expected_output`)
- Die GEval-Korrektheit von eben reiht sich einfach mit ein

In [34]:
import pandas as pd
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)

metrics = {
    "faithfulness": FaithfulnessMetric(model=judge_model, async_mode=False),
    "answer_relevancy": AnswerRelevancyMetric(model=judge_model, async_mode=False),
    "contextual_relevancy": ContextualRelevancyMetric(model=judge_model, async_mode=False),
    "contextual_recall": ContextualRecallMetric(model=judge_model, async_mode=False),
    "correctness": correctness,
}

In [35]:
rows = []
for case in [good_case, bad_case]: # Evaluate each test case
    row = {"answer": case.actual_output[:30] + "..."} # Initialize row with answer
    for name, metric in metrics.items(): # Evaluate each metric for the current test case
        metric.measure(case, _show_indicator=False) # _show_indicator=False is used to avoid showing the progress indicator in this example
        row[name] = metric.score # Store the score in the row dictionary, name is the metric name and score is the metric score
    rows.append(row)

pd.DataFrame(rows)

,answer,faithfulness,answer_relevancy,contextual_relevancy,contextual_recall,correctness
0,"Overfitting bedeutet, dass das...",1.0,1.0,1.0,1.0,0.7
1,Overfitting wird durch eine zu...,1.0,1.0,1.0,1.0,0.0



## Dieselben Fragen, andere Namen

Zwei Vokabulare, dieselben Konzepte:

| Was wird geprüft | Diese Woche (= RAGAS) | DeepEval |
|---|---|---|
| Abgerufene Docs nützlich? | Context Precision | Contextual Relevancy |
| Genug Kontext gefunden? | Context Recall | Contextual Recall |
| Antwort nur aus dem Kontext? | Faithfulness | Faithfulness |
| Antwort beantwortet die Frage? | Answer Relevancy | Answer Relevancy |
| Stimmt mit der Referenz überein? | Korrektheit (vs. Referenz) | GEval mit eigenem Kriterium |

RAGAS — das Framework, aus dem diese Metriken stammen — verwendet genau die
Namen dieser Woche; nur die beiden **Retrieval**-Metriken heißen bei
DeepEval anders. Lassen Sie sich von den Namen nicht verwirren